# Streamlit 
# 4. Mission Control Dashboard
We are in the finale. We have data (**Pandas**), we have charts (**Matplotlib**). Now we will combine it all into an interactive web application, needed for a clear overview and decision-making at headquarters.

What is `Streamlit`? It is a framework that turns your **Python script** into a beautiful **web application**. Without knowledge of HTML, CSS, or JavaScript.

**Overview & Roadmap:**
* Streamlit architecture and Data Flow principle (automatic Rerun).
* Dashboard layout (columns, sidebar, tabs).
* Data visualization using metrics (KPI), interactive tables, and charts.
* Interactivity and filters using widgets and forms.
* Data preservation using Session State.
* Modular architecture of project structuring (Pro tips).

## 4.0 Data Initialization (Simulation)
First, we must import the library. The standard alias shortcut for Streamlit in the community is `st`.

In [ ]:
import streamlit as st

## 4.1 App development

- Procedure:
    * in JPN we do the **dirty work** = preparation and testing of logic and functionality
    * we obtain, pre-prepare, clean, and format data
    * we prepare more complex charts (Matplotlib), leaving the simpler ones for Streamlit

#### **Streamlit** Key points:
1. **Frontend in Python:** Streamlit translates Python code into HTML/JS/CSS. We don't need to know web technologies.

2. **Server-Client model:** The application runs on a server (locally on our PC), the browser only displays it.

3. **The Rerun (Data Flow):** This is the most important concept = `any interaction` by the user (click, change) causes the `entire Python script to run again` from the first line to the last.
    - = `we lose variables` if we don't use Session State

4. Streamlit does not work in Jupyter Notebook but in `.py` files:
    * in IDE (VS Code) we create a file - e.g., `app.py`
    * we run the application (our python file) via the terminal

### Installation and Start

- in the terminal, we must navigate to the file / folder with app.py and then run with the command

```bash
streamlit run app.py
```

## 4.2 Application Skeleton (Layout & Config)
Every Streamlit application begins with page configuration - we put this at the beginning of the Streamlit code or in a separate file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Configuration ---
st.set_page_config(
    page_title="Nautilus Deep Dive",
    page_icon="🌊",
    layout="wide"
)

# --- 2. Sidebar (Navigation/Context) ---
st.sidebar.header("Nautilus-X")
st.sidebar.image("https://www.americanoceans.org/wp-content/uploads/2023/06/deep-submarine-1536x864.jpeg", caption="External Cam 1")
depth_target = st.sidebar.text_input("Target depth (m):", "11000")

# --- 3. Main Header ---
st.title("🌊 Deepwater Overview: Dashboard")
st.markdown("**Mission status:** MARIAN TRENCH DESCENT")
st.text("Salute to all explorers and striders !")
st.divider()

## 4.3. Data Display (Metrics and Tables)
The dashboard must show critical numbers (KPI) at first glance. Streamlit has the `st.metric` widget for this.

#### Data Simulation (Backend):
We will generate data from submarine sensors (depth, pressure, battery).

In [ ]:
data = {
    'time_min': np.arange(0, 60),
    'depth_m': np.linspace(0, 10900, 60), # Descent
    'pressure_bar': np.linspace(1, 1000, 60), # Pressure rises
    'battery_pct': np.linspace(100, 85, 60) - np.random.uniform(0, 2, 60) # Consumption
}
df_sub = pd.DataFrame(data)

# We split the screen into 4 columns for metrics / quick overviews
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(label="Current depth", value=f"{df_sub['depth_m'].iloc[-1]:.0f} m", delta="20 m/s")

with col2:
    st.metric(label="Outside pressure", value=f"{df_sub['pressure_bar'].iloc[-1]:.1f} bar", delta="15 bar")

with col3:
    st.metric(label="Battery", value=f"{df_sub['battery_pct'].iloc[-1]:.1f} %", delta="-0.5 %")

with col4:
    st.metric(label="Water temperature", value="1.2 °C", delta=None)

## 4.4. Visualization (Matplotlib Integration)
Here we connect **Streamlit** and **Matplotlib**. We will use the object-oriented approach (fig, ax) and pass the result to the `st.pyplot()` function.

#### Sonar:

In [ ]:
st.subheader("📉 Descent Telemetry")

# Creating chart - OOP approach
fig, ax = plt.subplots(figsize=(10, 4))

# Plotting: X-axis = Time, Y-axis = Pressure
ax.plot(df_sub['time_min'], df_sub['pressure_bar'], color='navy', linewidth=2)

# Styling
ax.set_title("Pressure increase")
ax.set_ylabel("Pressure (bar)")
ax.set_xlabel("Mission time (min)")
ax.grid(True, linestyle='--', alpha=0.3)
ax.set_facecolor('#f0f2f6') # Matching background with app

# Sending to Streamlit
st.pyplot(fig)

Streamlit has its own **built-in charts** that are interactive.

- for quick overviews, `responsive` streamlit charts (zoom, interactive) can be used
- we don't have control over settings and formats like with Matplotlib
- advantage: `prototyping` & `speed`

- basic chart types:
    1. st.line_chart() line chart
    2. st.bar_chart() bar chart
    3. st.area_chart() area chart
    4. st.scatter_chart() scatter chart
    5. st.map() scatter chart on map background

#### Automated Survey

In [ ]:
st.divider()
st.subheader("📡 Quick Overview (Native Charts)")

# A) Seismic Activity (Line Chart)
seismic_data = pd.DataFrame(
    np.random.randn(50, 2),
    columns=['Sensor A (Bow)', 'Sensor B (Stern)']
)
st.caption("Seismic Activity (Zoom enabled)")
st.line_chart(seismic_data) # plots columns as curves

# B) Submarine Location (Map)
st.caption("Submarine Location (Mariana Trench)")
map_data = pd.DataFrame(
    np.random.randn(1, 2) / [100, 100] + [11.35, 142.2], # Coordinates = columns 'lat' and 'lon'
    columns=['lat', 'lon']
)
st.map(map_data) # Plots map with points

## 4.5. Interactivity (Widgets & Filters)

Any change of a widget (slider, selectbox) immediately leads to a new execution of the application (from top to bottom).

#### Control Panel:

In [ ]:
st.sidebar.divider()
st.sidebar.subheader("History analysis")

# Widget: Slider for selecting time window
history_window = st.sidebar.slider("Display last X minutes:", min_value=5, max_value=60, value=20) # returns int

# DataFrame filtration based on widget
filtered_df = df_sub.tail(history_window)

# --- Dynamic Chart (Reacts to slider) ---
st.subheader(f"⚡ Energy consumption (Last {history_window} min)")

fig2, ax2 = plt.subplots(figsize=(10, 3))
ax2.plot(filtered_df['time_min'], filtered_df['battery_pct'], color='red', marker='o')
ax2.set_title("Battery charge")
ax2.set_ylim(80, 100) # Fixed Y-axis for better overview
ax2.grid(True)

st.pyplot(fig2)

## 4.6. Forms - Batch Input

**Critical concept**. Without forms, the application would "restart" (Rerun) after every character entered into an input or move of a slider. A form allows us to collect data and send it "in a batch" all at once.

#### Captain's Log

In [ ]:
st.divider()
st.header("📝 Captain's Log")

# 'with' block defines form area
with st.form(key='log_form'): # key='log_form' serves for identification
    st.write("New Mission Entry:")
    
    # Inputs inside form do not trigger Rerun
    officer = st.text_input("Officer Name", "Cmdr. Hornblower")
    log_message = st.text_area("Log Message", "Visual contact with unidentified fauna.")
    alert_level = st.select_slider("Alert Level", options=["Normal", "Warning", "Critical"])
    
    # Submit Button
    submit_button = st.form_submit_button(label="Save Entry") # action and app reload start only after submit

if submit_button: # action after submit
    st.success(f"Entry Saved: {officer} | Level: {alert_level}")
    st.write(f"Detail: {log_message}")

## 4.7. Session State (Individual App Memory)

**Critical concept**. Because Streamlit "restarts" the code with every action, local variables are cleared. If we want to keep something (e.g., sensor state, notes, current settings of anything), we must use `st.session_state`. Values will then remain preserved after interaction with the application and autoreload. We use this for `individual users`, where each session has its own state. If we refresh the application in the browser (F5), or close it = session state is lost.

#### System Switches:

In [ ]:
st.divider()
st.header("🛠️ Manual control")

# Initialization
if 'lights_on' not in st.session_state: # thanks to condition, runs only once at start
    st.session_state.lights_on = False # setting "session_state" variable

# Callback function
def toggle_lights():
    st.session_state.lights_on = not st.session_state.lights_on # negation of "session_state" variable

# Button triggers function
st.button("Main headlamps (ON/OFF)", on_click=toggle_lights) # function will be triggered "on click"

# Reaction to state
if st.session_state.lights_on:
    st.success("LIGHTS ON - Visibility 100%")
else:
    st.error("LIGHTS OFF - Silent Mode")

## 4.8. Caching (General Memory)

**Critical concept**. For cases of **heavy one-time operations** (loading DB, AI calculation, API data). We don't want to perform this on every Rerun. We use the `@st.cache_data` decorator. We use this for data that will be shared by `all users` of the application. Data is loaded only once and used from cache / memory during repeated function calls. "Time to live" of data in cache can be set using the ttl parameter.
import time

In [ ]:
import time


st.divider()
st.subheader("🧠 Onboard Computer (AI)")

# Function with caching = Streamlit remembers result for given inputs.
@st.cache_data(ttl=120) # ttl (time to live) in seconds
def calculate_trajectory(): # on next call, doesn't run function, just returns stored result
    time.sleep(3) # simulation of "demanding" calculation (3 seconds)
    return "Trajectory Calculated (Optimal Path Found)."

if st.button("Run Route Calculation"):
    # 1st click = waiting 3s
    # 2nd click = immediate result (from cache)
    result = calculate_trajectory()
    st.success(result)

## 4.9. Organization and Layout (Tabs & Expanders)

Tools for saving space on screen.

- `Tabs:` Logical division of modules.

- `Expander:` Hiding details/raw data.

In [ ]:
st.divider()
st.subheader("🎛️ System Modules")

# Creating 3 tabs
tab1, tab2, tab3 = st.tabs(["Bio-Scan", "Navigation", "Raw Data"])

with tab1:
    st.info("Scanning environment... No large lifeforms detected.")
    
with tab2:
    st.write("Sonar active. Seabed stable.")

with tab3:
    with st.expander("Show Full Telemetry Log"): # closed by default
        st.dataframe(df_sub) # data from previous section

## 4.10. Application Architecture (Refactoring)
As our application grows, the app.py file becomes cluttered. Professional practice dictates splitting code into logical units (modules).

Recommended project structure:

* `**utils.py (Backend):**` Helper functions, data loading, calculations. (Logic is here).

* `**views.py (Frontend components):**` Functions that render specific parts of the page (charts, sections).

* `**app.py (Main launcher):**` Only imports and assembles the application together.

**utils.py**
Move data loading and cached functions here.

In [ ]:
import pandas as pd
import numpy as np
import streamlit as st

@st.cache_data
def load_sensor_data():
    """
    Simulation of loading data from sensors.
    Returns DataFrame.
    """
    data = {
        'depth': np.linspace(0, 500, 100),
        'pressure': np.linspace(1, 50, 100),
        'temp': np.random.uniform(4, 2, 100)
    }
    return pd.DataFrame(data)

def calculate_oxygen_reserve(crew_size):
    """
    Calculation of remaining oxygen in hours.
    """
    base_reserve = 1000 # Units
    consumption_rate = 2.5 * crew_size
    return base_reserve / consumption_rate

**views.py**
Move code using st.write, st.plot etc. here. Each "page" or "section" will have its own function.

In [ ]:
import streamlit as st
import matplotlib.pyplot as plt

def show_dashboard_view(df):
    """
    Renders main overview (Dashboard).
    """
    st.header("📊 Main Control Deck")
    
    # Metrics
    col1, col2 = st.columns(2)
    col1.metric("Current Depth", f"{df['depth'].iloc[-1]:.0f} m")
    col2.metric("Water Temp", f"{df['temp'].iloc[-1]:.1f} °C")
    
    # Chart
    st.subheader("Pressure Trend")
    st.line_chart(df['pressure'])

def show_science_lab_view():
    """
    Renders laboratory section.
    """
    st.header("🔬 Biolab Module")
    st.info("Microscope active. Analyzing samples...")
    
    # Form for scientists
    with st.form("sample_form"):
        st.text_input("Sample ID")
        st.form_submit_button("Log Sample")

**app.py (Main Controller)**

Main file remains clean. Serves only as a router.

In [ ]:
import streamlit as st

# We import our own modules - must be in the same folder
from utils import load_sensor_data
from views import show_dashboard_view, show_science_lab_view

# 1. Configuration (Always first)
st.set_page_config(page_title="Nautilus Modular App", layout="wide")

# 2. Loading data (Backend)
df_sensors = load_sensor_data()

# 3. Navigation (Sidebar)
st.sidebar.title("Nautilus OS v2.0")
navigation = st.sidebar.radio("Go to:", ["Bridge (Dashboard)", "Science Lab"])

# 4. Router (Switching views)
if navigation == "Bridge (Dashboard)":
    show_dashboard_view(df_sensors) # We pass data to view function
    
elif navigation == "Science Lab":
    show_science_lab_view() # We display laboratory

# result is a short and readable file = easy maintenance

## Deployment
So far our application runs only on "simulator" (our computer = localhost). For command on Earth (anyone with link) to access it, we must launch it into orbit.

The easiest way is combination of GitHub + Streamlit Community Cloud.

### Step 1: Supply Manifest (requirements.txt)
The server where the application will run is a clean computer. It doesn't know it needs Pandas or Matplotlib. We must give it a list.

We create file `requirements.txt` - in active project venv run:

```bash
pip freeze > requirements.txt 
```

Omlouvám se, už rozumím. Problém vzniká při vnořování bloků kódu do sebe. Použiji nyní bezpečnější formátování (4 zpětné uvozovky pro obal), aby se vnitřní bloky (3 zpětné uvozovky) zobrazily správně a šlo to zkopírovat.

Zde je opravená část od Step 2 dále:

Markdown

### Step 2: Sending to GitHub (Uplink)
We upload our project to a new repository on GitHub. Repository must contain:

- `app.py` (eventually utils.py, views.py)

- `requirements.txt`

- any data files or images..

### Step 3: Broadcasting Activation (Streamlit Cloud)
Go to share.streamlit.io and log in with your GitHub account.

- Click on `"New App"`.

- Select our `repository`, branch (main) and main file (app.py).

- Click on `"Deploy!"`.

Verify that libraries are installing and our application is starting. Once done, you get a public URL address (e.g. ares-mission-control.streamlit.app).

## 🎉 Mission Accomplished: Final Debrief

Congratulations, cadets. You have completed the full training and successfully transformed your data into a format that even Mission Command can understand.

* What we achieved:

    * Left the Lab: Your code no longer sits idly in a Jupyter Notebook. We transformed it into standalone scripts (.py) that form a robust application.

    * Mastered Time: You grasped the Data Flow principle and how Streamlit "redraws" reality with every interaction. You learned to preserve critical information using Session State.

    * Unified Systems: You integrated the analytical power of Pandas and the precision of Matplotlib into a single interactive dashboard.

    * Launched the Satellite: Thanks to deployment via GitHub, your application is now accessible from anywhere in the universe.

Your digital infrastructure is ready. Now it is up to you. Ad Astra! 🚀

## Practise

### Quick System Test
We will create a simple "Sonar" application that reacts to user input (depth) and prints safety warnings.

**Assignment:**

- Create a new file sonar.py.

- Import streamlit.

- Add application title "Echelon Sonar".

- Create a slider for setting depth from 0 to 11 000 meters.

- Prepare a simple condition:

    - If depth is greater than 8000 m, display red message (st.error) "CRITICAL PRESSURE".

    - Otherwise display green message (st.success) "Systems Nominal".

- Run application in terminal with command `streamlit run sonar.py`.

## Homework (Project: Mars - Final Dashboard)

**Mission:** Command on Earth approved deployment of our analytical software. Our task is to create `Ares Mission Control Dashboard`, which integrates data (Pandas), advanced visualizations (Matplotlib) and quick overviews (Streamlit).

0. Input data: For this task we will use simulation based on data we processed in previous lessons.

```python
import pandas as pd
import numpy as np

def get_mars_mission_data():
    """Generates simulation data for the mission."""
    dates = pd.date_range(start='2035-01-01', periods=30, freq='D')
    df = pd.DataFrame({
        'date': dates,
        'sol': np.arange(1, 31),
        'avg_temp_c': np.random.uniform(-75, -55, 30),
        'pressure_pa': np.random.uniform(600, 620, 30),
        'radiation_rem': np.random.uniform(0.1, 0.5, 30),
        'power_output_kwh': np.linspace(100, 95, 30) - np.random.uniform(0, 5, 30) # Klesající výkon panelů
    })
    return df
```

### Assignment: Level 1 (The Monolith)
Create file app.py. All logic will be in this one file.

1. Configuration:

    - Set page title "Ares Mission Control" and layout `"wide"`.

    - Add main title and sidebar with Mars image.

2. Data & KPI:

    - Load data using function `get_mars_mission_data()`.

    - Create 3 columns and display metrics for last measured day: Sol, Temperature, Power.

3. Matplotlib Chart (Detailed Analysis):

    - Create object chart (fig, ax) displaying correlation between `Pressure (X)` and `Temperature (Y)`.

    - Use `scatter plot`.

    - Display chart in Streamlit.

4. Streamlit Native Chart (Quick Overview):

    - Display evolution of `Power` over time using native simple line chart.


### Assignment: Level 2 (The Architect)
Refactor application according to modularity principles. Split code into three files:

***utils.py (Backend):**

- Move function `get_mars_mission_data` here.

- Ensure data does not load again on every click.

**views.py (Frontend):**

- Create function `display_dashboard(df)`, which will contain code for rendering metrics and charts.

**app.py (Main):**

- Only page configuration, data loading from utils and calling function from views remains here.

---
#### © Jiří Svoboda (George Freedom)
- Web: https://GeorgeFreedom.com
- LinkedIn: https://www.linkedin.com/in/georgefreedom/
- Let's talk: https://cal.com/georgefreedom